# Exercises — Chapter 11: RegTech, Compliance, and Anti-Money Laundering

Complete the exercises from Lecture 11.

In [ ]:
# Your code here

## Data Lab — SEC EDGAR

Mine compliance-related language from bank annual reports to understand how regulatory obligations are disclosed over time.

In [ ]:
import requests, time, re, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

EDGAR_HEADERS = {"User-Agent": "LLM-Finance-Course your-email@example.com"}

def edgar_get(url):
    time.sleep(0.11)
    r = requests.get(url, headers=EDGAR_HEADERS, timeout=15)
    r.raise_for_status()
    return r.json()

def get_cik(ticker):
    data = edgar_get("https://www.sec.gov/files/company_tickers.json")
    for v in data.values():
        if v["ticker"].upper() == ticker.upper():
            return str(v["cik_str"]).zfill(10)
    raise ValueError(f"Ticker {ticker} not found")

def get_submissions(cik):
    return edgar_get(f"https://data.sec.gov/submissions/CIK{cik}.json")

def get_concept(cik, concept):
    return edgar_get(f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json")

def get_annual_series(cik, concept):
    data = get_concept(cik, concept)
    usd  = data.get("units", {}).get("USD", [])
    rows = [u for u in usd if u.get("form") == "10-K" and u.get("filed")]
    if not rows:
        for _, vals in data.get("units", {}).items():
            rows = [u for u in vals if u.get("form") == "10-K" and u.get("filed")]
            if rows: break
    df = (pd.DataFrame(rows)[["end","val","filed","accn"]]
            .rename(columns={"end":"date","val":concept})
            .drop_duplicates("date").sort_values("date"))
    df["date"] = pd.to_datetime(df["date"])
    return df

def fetch_10k_html(ticker):
    cik  = get_cik(ticker)
    subs = get_submissions(cik)
    f    = subs["filings"]["recent"]
    idx  = next(i for i, x in enumerate(f["form"]) if x == "10-K")
    acc  = f["accessionNumber"][idx].replace("-", "")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik.lstrip('0')}"
            f"/{acc}/{f['primaryDocument'][idx]}")
    time.sleep(0.15)
    return requests.get(url, headers=EDGAR_HEADERS, timeout=30).text

def extract_mda(html):
    text = re.sub(r"<[^>]+>", " ", html)
    m    = re.search(
        r"Item\s+7[.\s]+Management.{0,80}Discussion.*?(?=Item\s+7A|Item\s+8)",
        text, re.IGNORECASE | re.DOTALL)
    raw  = m.group(0) if m else text[:30000]
    return re.sub(r"\s+", " ", raw).strip()

LM_POS = {"profitable","strong","growth","improved","exceeded","innovative",
           "efficient","leading","record","gained","successful","robust",
           "increased","advancing","outperformed","benefit","enhanced","positive","achieved","favourable"}
LM_NEG = {"risk","loss","uncertain","decline","adverse","default","impair",
           "volatile","litigation","breach","failed","reduced","weakness",
           "debt","distress","penalty","violation","exposure","downturn","writedown"}


### Exercise [B]: Regulatory Keyword Monitor

In [ ]:
# --- Exercise [B]: Regulatory Keyword Monitor ---
COMPLIANCE_KW = ["anti-money laundering","aml","bsa","ofac","sanctions",
                 "kyc","suspicious activity","fincen","compliance programme"]

cik_jpm = get_cik("JPM")
subs_j  = get_submissions(cik_jpm)
fj      = subs_j["filings"]["recent"]
k10_idx = [i for i, x in enumerate(fj["form"]) if x == "10-K"][:5]

rows_11b = []
for i in k10_idx:
    year = int(fj["filingDate"][i][:4])
    acc  = fj["accessionNumber"][i].replace("-","")
    url  = (f"https://www.sec.gov/Archives/edgar/data/{cik_jpm.lstrip('0')}"
            f"/{acc}/{fj['primaryDocument'][i]}")
    time.sleep(0.15)
    raw  = requests.get(url, headers=EDGAR_HEADERS, timeout=30).text
    text = re.sub(r"<[^>]+>", " ", raw).lower()
    words = len(re.findall(r"[a-z]+", text))
    row  = {"year": year}
    for kw in COMPLIANCE_KW:
        row[kw] = text.count(kw) * 1000 / max(words,1)
    rows_11b.append(row)
    print(f"{year}: {sum(row[kw] for kw in COMPLIANCE_KW):.3f} kw/1000 words")

df_11b = pd.DataFrame(rows_11b).set_index("year").sort_index()
print(df_11b.round(3))
total_by_year = df_11b.sum(axis=1)
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(total_by_year.index, total_by_year.values, "o-", color="steelblue", linewidth=2)
ax.set_xlabel("Year"); ax.set_ylabel("Total compliance keywords per 1 000 words")
ax.set_title("JPM 10-K: Compliance Keyword Frequency Over Time")
plt.tight_layout(); plt.show()

### Exercise [I]: Risk-Factor Paragraph Classification

In [ ]:
# --- Exercise [I]: Risk-Factor Paragraph Classification ---
RISK_RULES = {
    "credit risk":               ["credit","loan","default","counterparty","impairment","allowance"],
    "regulatory/compliance risk":["regulatory","compliance","sec","fed","fdic","aml","sanctions","enforcement"],
    "operational risk":          ["operational","technology","cyber","system","fraud","breach","continuity"],
    "market risk":               ["market","interest rate","liquidity","volatility","currency","hedg"],
    "other":                     [],
}

cik_jpm = get_cik("JPM")
html_jpm = fetch_10k_html("JPM")
text_all  = re.sub(r"<[^>]+>", " ", html_jpm).lower()
risk_m = re.search(r"item\s+1a.*?(?=item\s+1b|item\s+2)", text_all, re.DOTALL)
risk_text = risk_m.group(0) if risk_m else text_all[:50000]
paras = [p.strip() for p in re.split(r"\n{2,}|\. {3,}", risk_text) if len(p.split()) >= 20]

def classify(para):
    para_l = para.lower()
    for cat, kws in RISK_RULES.items():
        if cat == "other": continue
        if any(kw in para_l for kw in kws): return cat
    return "other"

labels_11i = [classify(p) for p in paras]
from collections import Counter
dist = Counter(labels_11i)
print("Class distribution:", dict(dist))

for cat in RISK_RULES:
    sample = [p for p, l in zip(paras, labels_11i) if l == cat]
    if sample: print(f"\n[{cat.upper()}]\n{sample[0][:300]}...")

### Exercise [A]: Cross-Bank Compliance Language Comparison

In [ ]:
# --- Exercise [A]: Cross-Bank Comparison with Clustering ---
from scipy.cluster.hierarchy import linkage, dendrogram

BANKS_11A = ["JPM","BAC","C","WFC","GS"]
matrix_rows = []
for t in BANKS_11A:
    try:
        cik_b = get_cik(t)
        subs_b = get_submissions(cik_b)
        fb = subs_b["filings"]["recent"]
        idx = next(i for i, x in enumerate(fb["form"]) if x == "10-K")
        acc = fb["accessionNumber"][idx].replace("-","")
        url = (f"https://www.sec.gov/Archives/edgar/data/{cik_b.lstrip('0')}"
               f"/{acc}/{fb['primaryDocument'][idx]}")
        time.sleep(0.15)
        raw  = requests.get(url, headers=EDGAR_HEADERS, timeout=30).text
        text = re.sub(r"<[^>]+>", " ", raw).lower()
        words = len(re.findall(r"[a-z]+", text))
        row  = {kw: text.count(kw)*1000/max(words,1) for kw in COMPLIANCE_KW}
        matrix_rows.append(row)
        print(f"{t}: done")
    except Exception as e:
        print(f"{t}: {e}"); matrix_rows.append({kw:0 for kw in COMPLIANCE_KW})

df_matrix = pd.DataFrame(matrix_rows, index=BANKS_11A)[COMPLIANCE_KW]
print(df_matrix.round(3))

Z = linkage(df_matrix.values, method="ward")
fig, ax = plt.subplots(figsize=(7,4))
dendrogram(Z, labels=BANKS_11A, ax=ax)
ax.set_title("Compliance Language Dendrogram — Large US Banks")
plt.tight_layout(); plt.show()